# 10 - Stacking multimodal con ajuste de hiperparametros

Notebook de ensemble avanzado con StackingClassifier y busqueda exhaustiva sobre la capa meta.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import f1_score, classification_report

In [2]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones'
WEIGHTS_DIR = PROJECT_ROOT / 'weights_hpo'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, WEIGHTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

bundle = np.load(ARTIFACTS_DIR / 'fusion_features.npz', allow_pickle=True)
X_all = bundle['X_fusion'].astype(np.float32)
y_all = bundle['y'].astype(np.int64)
langs_all = bundle['langs'].astype(str)
sample_ids = bundle['sample_ids'].astype(str)
print('X_all:', X_all.shape, '| etiquetas validas:', int((y_all >= 0).sum()))

X_all: (2006, 884) | etiquetas validas: 2006


In [3]:
def build_stacking():
    base_estimators = [
        ('svm', Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, class_weight='balanced', C=10.0, gamma='scale', random_state=42))
        ])),
        ('rf', RandomForestClassifier(
            n_estimators=600, max_depth=None, min_samples_leaf=1,
            class_weight='balanced_subsample', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', n_jobs=-1, random_state=42))
        ]))
    ]

    meta = LogisticRegression(max_iter=4000, class_weight='balanced', n_jobs=-1, random_state=42)
    stack = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta,
        stack_method='predict_proba',
        cv=5,
        passthrough=False,
        n_jobs=-1
    )
    return stack

In [4]:
def export_json(sample_ids, pred_binary, output_path):
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': 'YES' if int(v) == 1 else 'NO'}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)
    print('Exportado:', output_path, '| filas:', len(rows))

def run_stacking(tag, train_langs):
    mask = np.isin(langs_all, train_langs) & (y_all >= 0)
    X_train = X_all[mask]
    y_train = y_all[mask]

    if len(np.unique(y_train)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    stack = build_stacking()
    param_grid = {
        'final_estimator__C': [0.1, 1.0, 5.0, 10.0],
        'passthrough': [False, True]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(
        estimator=stack,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    gs.fit(X_train, y_train)

    print(tag, 'best CV f1_macro:', gs.best_score_)
    print('best params:', gs.best_params_)

    cv_rows = []
    for i in range(len(gs.cv_results_['params'])):
        cv_rows.append({
            'config': tag,
            'mean_test_score': float(gs.cv_results_['mean_test_score'][i]),
            'std_test_score': float(gs.cv_results_['std_test_score'][i]),
            **{f'param_{k}': v for k, v in gs.cv_results_['params'][i].items()}
        })
    cv_df = pd.DataFrame(cv_rows).sort_values('mean_test_score', ascending=False)
    cv_path = REPORTS_DIR / f'stacking_cv_{tag}.csv'
    cv_df.to_csv(cv_path, index=False)
    print('Reporte CV:', cv_path)

    best_model = gs.best_estimator_
    pred_train = best_model.predict(X_train)
    print('Train macro-f1:', f1_score(y_train, pred_train, average='macro'))
    print(classification_report(y_train, pred_train, digits=4))

    model_path = WEIGHTS_DIR / f'stacking_best_{tag}.joblib'
    joblib.dump(best_model, model_path)
    print('Modelo guardado:', model_path)

    pred_all = best_model.predict(X_all)
    out_path = PRED_DIR / f'BeingChillingWeWillWin_STACK_{tag}.json'
    export_json(sample_ids, pred_all, out_path)

run_stacking('ES', ['es'])
run_stacking('EN', ['en'])
run_stacking('ES_EN', ['es', 'en'])

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
 

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
 

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
 

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
 

...

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 4000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=4000).
You might also want to scale the data as shown in:
 

ES_EN best CV f1_macro: 0.6663145365934504
best params: {'final_estimator__C': 1.0, 'passthrough': False}
Reporte CV: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/reports_hpo/stacking_cv_ES_EN.csv


Train macro-f1: 1.0
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      1045
           1     1.0000    1.0000    1.0000       961

    accuracy                         1.0000      2006
   macro avg     1.0000    1.0000    1.0000      2006
weighted avg     1.0000    1.0000    1.0000      2006



Modelo guardado: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/weights_hpo/stacking_best_ES_EN.joblib


Exportado: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/predicciones/BeingChillingWeWillWin_STACK_ES_EN.json | filas: 2006
